# 课程 5 选做 · embedding 的底层实现

这里是从主课移出来的内容：shape 推导、one-hot 与查表等价的代码证明、改表行定位、显式 one-hot 的内存账、补齐后的平均。更适合准备算法或模型工程方向，解决方案面试一般不考。主课和作业都不依赖这里。

积分记在主课同一个本地记录里（另有 80 分），不计入必做。按顺序运行，不要点“全部运行”，因为 TODO 故意未填写。

In [ ]:
from pathlib import Path
import sys
import torch
import torch.nn.functional as F

candidates = [Path.cwd() / "05-token-embedding", Path.cwd()]
COURSE_DIR = next((p for p in candidates if (p / "tokenizer_tools.py").exists()), None)
if COURSE_DIR is None:
    raise RuntimeError("请从项目根目录或 05-token-embedding 目录运行。")
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))
from IPython.display import display
from tokenizer_tools import load_tokenizer
from learning_lab import LessonLab

tokenizer = load_tokenizer()
torch.set_printoptions(precision=3, sci_mode=False)
lab = LessonLab(tokenizer, COURSE_DIR / ".learning-progress.json")
print("Python:", sys.executable)
display(lab.dashboard)

In [ ]:
toy_vocab = ["猫", "追", "狗", "，", "看", "星", "空", "。"]
V, D = len(toy_vocab), 4
generator = torch.Generator(device="cpu").manual_seed(7)
E = torch.randn(V, D, generator=generator, dtype=torch.float32, device="cpu")
ids = torch.tensor([0, 1, 2, 3, 0, 4, 5, 6, 7], dtype=torch.long, device="cpu")

print("ID顺序:", ids.tolist())
print("对应文字:", "".join(toy_vocab[i] for i in ids.tolist()))
print("E形状:", tuple(E.shape))
print(E)


改表行的检查需要先记录未修改的表，所以先运行一次矩阵实验台。

In [ ]:
display(lab.embedding(E, ids, toy_vocab))


### 1．先预测形状，再证明两种方法等价

先填：

| 对象 | 你预测的 shape | 各维表示什么 |
|---|---|---|
| `ids` | ____ | ____ |
| `E[ids]` | ____ | ____ |
| one-hot，行对应位置、列对应词表 | ____ | ____ |
| one-hot 与 E 相乘的结果 | ____ | ____ |

对比你以前问过的整型索引与切片：`E[0]` 取第 0 行，结果没有行轴；`E[0:1]` 也只取一行，但保留行轴。这里的 0 是编号为 0，**不是“取零个”**。


In [ ]:
print("E[0] shape:", tuple(E[0].shape), "值:", E[0])
print("E[0:1] shape:", tuple(E[0:1].shape), "值:", E[0:1])
print("E[ids] shape:", tuple(E[ids].shape))


下面可以观察 ids、E 和 E[ids]。选一个对象，分别用整数和切片取一行，检查数字与 shape；再回到上方表格补充你的解释。观察不会替你完成后面的两种实现。



In [ ]:
from lab_visuals import TensorExplorer

# 传入课堂实际张量；这里的轴名称也来自它们在本实验中的用途。
tensor_scope = TensorExplorer({
    "ids": (ids, ["输入位置 T"]),
    "E": (E, ["词表项 V", "向量维度 D"]),
    "E[ids]": (E[ids], ["输入位置 T", "向量维度 D"]),
})
display(tensor_scope.widget)


**任务 2：实现两种 embedding 计算并比较。**

写 `embedding_two_ways(E, ids)`，返回 `by_lookup, by_matmul`：
1. 直接按 ID 取行。
2. 构造 one-hot 后与 E 相乘。

API 提示：

```python
example_ids = torch.tensor([1, 0], dtype=torch.long)
F.one_hot(example_ids, num_classes=3)  # 两行、三列；每行只有一处为1
some_tensor.to(torch.float32)         # 转成float32，返回转换结果
torch.testing.assert_close(a, b)      # 数值不接近时会报错
```

`F.one_hot` 默认产生 int64；和浮点表做矩阵乘法前，需要转成 `E.dtype`。词表行数从 `E.shape[0]` 读取，不在函数里写死 8。说明函数为什么等价，比看到一个“通过”更重要。


<details><summary>线索 1：我们只盯住 one-hot 的一行</summary>

这一行只有一个 1，其余都是 0。把它与 E 相乘：每一项分别让 E 的哪一行留下，哪一行消失？先说清这个动作，再写代码。

</details>

<details><summary>线索 2：沿着轴检查连接处</summary>

one-hot 的最后一维要与 E 的行数相同。乘完以后，保留的是输入长度 T 和向量维度 D。你直接查表时，是否也为每个输入位置拿回了 D 个数？

</details>

<details><summary>局部例子：只看一行乘法</summary>

对于三行的表，`[0, 1, 0]` 与它相乘时，对应的是 `0 × 第0行 + 1 × 第1行 + 0 × 第2行`。先解释剩下什么，再把这个道理扩展到你的整组 ids。

</details>


In [ ]:
def embedding_two_ways(E, ids):
    # TODO 2：完成查表、one-hot矩阵乘法，返回两个张量。
    raise NotImplementedError("请完成 embedding_two_ways()。")

by_lookup, by_matmul = embedding_two_ways(E, ids)
assert by_lookup.shape == (len(ids), D)
torch.testing.assert_close(by_lookup, by_matmul)
print("两种结果一致。第一、第五个位置是否取了同一行:", torch.equal(by_lookup[0], by_lookup[4]))


In [ ]:
display(lab.checkpoint("lookup", lambda: embedding_two_ways))


**解释，不只写“结果相同”：**

- one-hot 这一行中，0 和 1 分别让哪些项留下或消失？____。
- T、V、D 分别由什么决定？把 D 从 4 换成 6，输出哪一维改变？____。
- 同一个 ID 在两个位置取到相同向量，能否说明后续网络中的表示也一定相同？先写当前实验能证明的范围：____。

直接查表与 one-hot 矩阵乘是数学上的等价写法；实际 embedding 层可以直接取行，不必先造出大多数元素为 0 的矩阵。


### 2．换个条件：修改一行，哪些位置应该改变？

仍用上面的 E 和 ids。只把 ID=0 对应的那一行增加一个向量。先写出你预测变化的位置编号（位置从 0 开始）：____。

**任务 3：完成一次改动，并验证自己的预测。**

用到的写法：`E.clone()` 得到独立副本，不能只写 `changed_E = E`；否则两个变量指向同一张量。逐个位置用 `torch.equal(before, after)` 判断向量是否相同即可，不要求使用新的高级索引技巧。


In [ ]:
changed_E = E.clone()
target_id = 0
delta = torch.tensor([1.0, -2.0, 0.5, 3.0])

# TODO 3a：只修改 target_id 对应的那一行。
raise NotImplementedError("请完成改行操作，并删除此行。")

before = E[ids]
after = changed_E[ids]
# TODO 3b：找出发生变化的位置，填入 changed_positions（用循环即可）。
changed_positions = None
assert changed_positions is not None, "请比较每个位置的前后向量。"
print("实际变化的位置:", changed_positions)
print("这些位置的 token ID:", ids[changed_positions].tolist())


In [ ]:
display(lab.checkpoint("row", lambda: (E, ids, changed_E, changed_positions, target_id, delta)))


**我的解释：**

- 预测与实测是否一致？对应的是哪个 token 的哪些出现位置？____。
- 为什么只改一行会影响多个位置？____。
- 如果把这张表中的全部 ID 重新编号，同时调整输入 ID，查到的向量能否保持不变？这个思考说明 ID 数值能否直接代表含义？____。

本课还没有位置编码和 attention。同一个 ID 的**初始查表结果**相同；经过后续上下文处理后，同一 token 在不同上下文中的表示可以不同。下一课会用小矩阵观察它怎样发生。


<details><summary>接回真实分词器：表需要多少行？</summary>

### 3．接回真实 token ID，机制是否改变？

把真实分词器产生的 ID 用在一张**新随机表**里。D 只取 8，所以表很小；它仍不是下载的 Qwen 模型参数。这里不创建大 one-hot。

表至少需要覆盖所有可能的 ID。读取词表中的最大 ID 再加 1，避免凭模型名字猜行数。注意：实际模型配置可能为 embedding 预留额外行，分词器已登记项数不一定等于某个模型权重的行数。

</details>


In [ ]:
real_ids_list = tokenizer.encode("猫追狗，猫看星空。", add_special_tokens=False).ids
real_ids = torch.tensor(real_ids_list, dtype=torch.long)
all_vocab = tokenizer.get_vocab(with_added_tokens=True)
real_V = max(all_vocab.values()) + 1
real_D = 8
real_E = torch.randn(real_V, real_D, generator=generator, dtype=torch.float32, device="cpu")
real_vectors = real_E[real_ids]

print("真实 ID:", real_ids_list)
print("自建随机表 shape:", tuple(real_E.shape))
print("查表输出 shape:", tuple(real_vectors.shape))
print("表的实际字节数:", real_E.numel() * real_E.element_size())


In [ ]:
# 表很大时只展示有限窗口；shape 与数据量来自完整实际对象。
real_scope = TensorExplorer({
    "real_ids": (real_ids, ["输入位置 T"]),
    "real_E": (real_E, ["词表行 V", "向量维度 D"]),
    "real_vectors": (real_vectors, ["输入位置 T", "向量维度 D"]),
})
display(real_scope.widget)


### 先试：同一段文字，为什么一种算法放不下？

下方默认只给这些张量 **2 GB 假设额度**。先观察两种方案，再保持 V、D 不变，把 T 从 2048 调到 4096：哪些部分变了？恢复 T 后，只改 D，再观察一次。拖滑块不计分，后面的估算函数和解释才用于验证理解。


In [ ]:
display(lab.memory())


### 4．不能只说“查表节省内存”：算出节省了哪一项

我们把文本长度翻倍。表里的词表项有没有因此变多？把这件事想清楚，再写公式，就不容易把共享的表重复算进去。

给定一个假设配置：B=4 条文本，每条补齐到 T=2048 个位置，V=150000 行词表，D=768。

- ID 张量为 `[B,T]`，int64 每项 8 字节。
- 如果显式创建 float32 one-hot，为 `[B,T,V]`，每项 4 字节。
- E 表为 `[V,D]`，假设 float32。
- 输出为 `[B,T,D]`，假设 float32。

**两种方法都要用 E 表，也都要产生输出。** 比较时不能把 E 只算给查表方案。这里估算这些张量的原始数据体积，不是完整运行峰值，也不含梯度、优化器、缓存、临时副本等。

**任务 4：实现 `representation_bytes(B,T,V,D)`**，返回四个字段：`ids`、`one_hot`、`table`、`output`，每个值单位为字节。本题仅做乘法计算，**不要真正分配这些大张量**。


<details><summary>线索 1：我们先数元素，再算字节</summary>

把每个张量的 shape 写出来。各维相乘得到元素个数，再乘每个元素的字节数。函数返回字节；展示 GB 时，再除以 10⁹。

</details>

<details><summary>线索 2：共享的东西放在同一本账里</summary>

两种方案都用 E 表，也都要输出向量。直接查表避免的是显式 one-hot，不是让表和输出消失。F.one_hot 先产生 int64，再转 float32 时还可能保留临时副本，所以本题的 float32 账不是完整运行峰值。

</details>

<details><summary>局部例子：检查单位有没有丢</summary>

假设一张 `[2,3]` 的表，每个元素占 4 字节，那么原始数据是 `2 × 3 × 4 = 24 bytes`。现在分别写出本题四个张量的轴，别把这张小表的公式直接套给所有对象。

</details>


In [ ]:
def representation_bytes(B, T, V, D):
    # TODO 4：返回 ids、one_hot、table、output 四项的字节数。
    raise NotImplementedError("请按各张量形状计算。")

sizes = representation_bytes(B=4, T=2048, V=150_000, D=768)
for name, size_bytes in sizes.items():
    print(name, ":", size_bytes, "bytes =", round(size_bytes / 1_000_000_000, 6), "GB")
print("显式float32 one-hot / int64 IDs体积比:", sizes["one_hot"] / sizes["ids"])


In [ ]:
display(lab.checkpoint("memory", lambda: representation_bytes))


**我的数量结论：**

1. 这组配置，ID 为 ____ GB，one-hot 为 ____ GB，表为 ____ GB，输出为 ____ GB。
2. 直接查表主要避免了哪一项，哪些是两种方式共享的？____。
3. 如果 T 翻倍，哪些项翻倍，哪一项不变？如果 D 翻倍呢？____。
4. 为什么上面不能直接称为“完整模型需要的内存”？____。

这题沿用你已经会的参数内存换算，提高的是计算范围与条件变化的判断，而不是增加算术难度。


![三维张量可以先固定批次轴，再观察一张二维表](assets/lesson5-batch.svg)



## 5 · 补齐长度后，平均值为什么变了？

两句话长度不同，放进同一个矩形张量前需要补齐。为了观察错误，我们故意用 0 补齐，但本课词表中 ID=0 本身又是合法的“猫”。

因此 `ids != 0` 不能区分“真猫”和“补齐位置”。需要另存 `valid_mask`：True 表示原文有效位置，False 表示补齐。它按原始长度生成。

下面的 mask 只用于本实验排除补齐项，**不是下一课的因果遮挡**。


In [ ]:
batch_ids = torch.tensor([[0, 1, 2, 0, 0], [0, 1, 2, 3, 0]], dtype=torch.long)
valid_mask = torch.tensor([[True, True, True, False, False],
                           [True, True, True, True, True]])
batch_vectors = E[batch_ids]
print("ids / mask / vectors shape:", batch_ids.shape, valid_mask.shape, batch_vectors.shape)
print("直接平均:", batch_vectors.mean(dim=1))
print("第一句话原本的平均:", E[torch.tensor([0, 1, 2])].mean(dim=0))


In [ ]:
batch_scope = TensorExplorer({
    "batch_vectors": (batch_vectors, ["批次 B", "输入位置 T", "向量维度 D"]),
    "batch_ids": (batch_ids, ["批次 B", "输入位置 T"]),
    "valid_mask": (valid_mask, ["批次 B", "输入位置 T"]),
})
display(batch_scope.widget)


**挑战：实现只对有效位置求平均的函数。**

输入为 `vectors[B,T,D]` 与 `mask[B,T]`，返回 `[B,D]`。如果任何一条文本没有有效位置，抛出 `ValueError`，不要除以 0。

API：
- `mask.unsqueeze(-1)` 在末尾加一个轴，得到 `[B,T,1]`，乘向量时广播到 D。
- `x.sum(dim=1)` 沿时间/位置轴求和。
- `raise ValueError("没有有效位置")` 明确停止非法输入。

<details><summary>提示 1</summary>
有效位置求和，再除以有效位置个数。分母不是补齐后的 T。
</details>

<details><summary>提示 2</summary>
把 mask 转成向量的 dtype；检查每条的有效数大于 0。使用 [B,T,1] mask 可以让求和后的分子 [B,D] 和分母 [B,1] 正确相除。
</details>


In [ ]:
RUN_CHALLENGE = False  # 想做挑战时改为 True；默认不阻塞必做部分。

def masked_mean(vectors, mask):
    # 可选 TODO：求有效位置的均值，并明确处理空文本。
    raise NotImplementedError("请完成可选 masked_mean()。")

if RUN_CHALLENGE:
    pooled = masked_mean(batch_vectors, valid_mask)
    expected = E[torch.tensor([0, 1, 2])].mean(dim=0)
    torch.testing.assert_close(pooled[0], expected)
    torch.testing.assert_close(pooled[1], E[batch_ids[1]].mean(dim=0))
    try:
        masked_mean(batch_vectors[:1], torch.zeros_like(valid_mask[:1]))
    except ValueError:
        print("没有有效位置时，已明确报错。")
    else:
        raise AssertionError("全空mask应抛出ValueError，不能返回无意义的均值。")
    longer_ids = torch.cat([batch_ids, torch.zeros((2, 3), dtype=torch.long)], dim=1)
    longer_mask = torch.cat([valid_mask, torch.zeros((2, 3), dtype=torch.bool)], dim=1)
    torch.testing.assert_close(pooled, masked_mean(E[longer_ids], longer_mask))
    print("增加补齐位置后，有效内容的均值不变。")


In [ ]:
if RUN_CHALLENGE:
    display(lab.checkpoint("mask", lambda: masked_mean))


做完后回到 [主课](lesson.ipynb)。这里的代码通过检查，同样只说明这些用例成立；原因解释需要讨论。